In [0]:
%sql
CREATE VOLUME IF NOT EXISTS main/Volumes/main/default/assessment_files.default.assessment_files;

In [0]:

# Databricks notebook source
# Azure Databricks Assessment - City Water Network Monitoring
# SETUP ONLY: this notebook creates tables, not exercise solutions.
# Upload the five data files into the Unity Catalog Volume folder before running.

# COMMAND ----------

import re
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType, TimestampType
)

dbutils.widgets.text("catalog", "main")
dbutils.widgets.text("schema", "default")
dbutils.widgets.text("volume", "assessment_files")
dbutils.widgets.text("folder", "water_network")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
volume = dbutils.widgets.get("volume").strip()
folder = dbutils.widgets.get("folder").strip().strip("/")

for label, value in [("catalog", catalog), ("schema", schema), ("volume", volume)]:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value):
        raise ValueError(f"Invalid {label}: use letters, digits and underscores only")
if not re.fullmatch(r"[A-Za-z0-9_/-]+", folder) or ".." in folder:
    raise ValueError("Invalid folder name")

namespace = f"`{catalog}`.`{schema}`"
base_path = f"/Volumes/{catalog}/{schema}/{volume}/{folder}"
print("Data location:", base_path)

required = [
    "water_meter_readings.csv", "water_zones.json", "water_service_requests.csv",
    "water_reading_changes.csv", "water_readings_incremental.csv"
]
try:
    available = {entry.name.rstrip("/") for entry in dbutils.fs.ls(base_path)}
except Exception as exc:
    raise RuntimeError(f"Cannot access {base_path}. Create the Volume and upload the files first.") from exc
missing = sorted(set(required) - available)
if missing:
    raise FileNotFoundError(f"Missing data files in {base_path}: {missing}")

spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

def save_delta(df, name):
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{catalog}.{schema}.{name}"))

# COMMAND ----------

reading_schema = StructType([
    StructField("reading_id", IntegerType(), True),
    StructField("meter_id", StringType(), True),
    StructField("zone_id", StringType(), True),
    StructField("reading_time", TimestampType(), True),
    StructField("flow_lpm", DoubleType(), True),
    StructField("pressure_bar", DoubleType(), True),
    StructField("chlorine_mg_l", DoubleType(), True),
    StructField("consumption_liters", IntegerType(), True),
    StructField("reading_status", StringType(), True),
    StructField("source_system", StringType(), True),
])

def read_readings(filename):
    return (spark.read.format("csv")
        .option("header", "true")
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
        .schema(reading_schema)
        .load(f"{base_path}/{filename}"))

source = read_readings("water_meter_readings.csv")
save_delta(source, "water_meter_readings_source")
save_delta(source, "water_meter_readings_work")
save_delta(read_readings("water_reading_changes.csv"), "water_reading_changes")
save_delta(read_readings("water_readings_incremental.csv"), "water_readings_incremental_staging")

# COMMAND ----------

zone_raw = spark.read.json(f"{base_path}/water_zones.json")
save_delta(zone_raw, "water_zones_raw")
zone_flat = zone_raw.select(
    "zone_id", "zone_name",
    F.col("location.city").alias("city"),
    F.col("location.district").alias("district"),
    "zone_type", "daily_capacity_liters", "active", "commissioned_on"
)
save_delta(zone_flat, "water_zones")

# COMMAND ----------

request_schema = StructType([
    StructField("request_id", IntegerType(), True),
    StructField("meter_id", StringType(), True),
    StructField("zone_id", StringType(), True),
    StructField("reported_at", TimestampType(), True),
    StructField("issue_type", StringType(), True),
    StructField("priority", StringType(), True),
    StructField("estimated_cost", DoubleType(), True),
    StructField("work_status", StringType(), True),
])
requests = (spark.read.option("header", "true")
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
    .schema(request_schema)
    .csv(f"{base_path}/water_service_requests.csv"))
save_delta(requests, "water_service_requests")

# COMMAND ----------

for name in [
    "water_meter_readings_source", "water_meter_readings_work",
    "water_zones_raw", "water_zones", "water_service_requests",
    "water_reading_changes", "water_readings_incremental_staging"
]:
    print(f"{catalog}.{schema}.{name}: {spark.table(f'{catalog}.{schema}.{name}').count()} rows")
print("SETUP COMPLETE. Do not rerun after beginning exercises or pipeline ingestion.")

Data location: /Volumes/main/default/assessment_files/water_network
main.default.water_meter_readings_source: 157 rows
main.default.water_meter_readings_work: 157 rows
main.default.water_zones_raw: 11 rows
main.default.water_zones: 11 rows
main.default.water_service_requests: 36 rows
main.default.water_reading_changes: 12 rows
main.default.water_readings_incremental_staging: 18 rows
SETUP COMPLETE. Do not rerun after beginning exercises or pipeline ingestion.


In [0]:
%sql
create or replace table water_readings_work
using delta
as 
select * from water_meter_readings_source;

num_affected_rows,num_inserted_rows


Section A | SQL Querying and Filtering

In [0]:
#1
spark.table('water_meter_readings_source').printSchema()
spark.table('water_meter_readings_source').count()

root
 |-- reading_id: integer (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- reading_time: timestamp (nullable = true)
 |-- flow_lpm: double (nullable = true)
 |-- pressure_bar: double (nullable = true)
 |-- chlorine_mg_l: double (nullable = true)
 |-- consumption_liters: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- source_system: string (nullable = true)



157

In [0]:
#2
from pyspark.sql import functions as F
filtering = spark.table('water_meter_readings_source').filter(
    F.col('reading_status')=='Valid'
).select(
    'reading_id',
    'meter_id',
    'zone_id',
    'reading_time',
    'consumption_liters',
)
filtering.show()

+----------+--------+-------+-------------------+------------------+
|reading_id|meter_id|zone_id|       reading_time|consumption_liters|
+----------+--------+-------+-------------------+------------------+
|      1001|    M001|    W01|2026-10-01 07:00:00|              2365|
|      1002|    M002|    W02|2026-10-01 08:10:00|              3545|
|      1003|    M003|    W03|2026-10-01 09:20:00|               548|
|      1004|    M004|    W04|2026-10-01 10:00:00|              2871|
|      1006|    M006|    W06|2026-10-01 12:20:00|              2171|
|      1007|    M007|    W07|2026-10-01 13:00:00|              4167|
|      1008|    M008|    W08|2026-10-01 14:10:00|              3621|
|      1009|    M009|    W09|2026-10-01 15:20:00|              1609|
|      1010|    M010|    W10|2026-10-01 16:00:00|              1829|
|      1011|    M011|    W01|2026-10-02 07:10:00|               313|
|      1013|    M013|    W03|2026-10-02 09:00:00|              2071|
|      1014|    M014|    W04|2026-

In [0]:
#3
from pyspark.sql import functions as F
pressure_bar = spark.table('water_meter_readings_source').filter(
    F.col('pressure_bar') < 2.5
).orderBy(F.col('pressure_bar').asc())
pressure_bar.show()

+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+
|reading_id|meter_id|zone_id|       reading_time|flow_lpm|pressure_bar|chlorine_mg_l|consumption_liters|reading_status|source_system|
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+
|      1071|    M011|    W01|2026-10-08 07:10:00|    10.9|         1.8|          0.8|               647|         Valid|   SmartMeter|
|      1064|    M004|    W04|2026-10-07 10:00:00|    20.3|        1.82|         0.84|               708|         Valid|      Gateway|
|      1024|    M024|    W04|2026-10-03 10:20:00|    44.5|        1.83|         1.25|              2085|         Valid|     FieldApp|
|      1116|    M026|    W06|2026-10-12 12:10:00|    28.9|        1.84|         0.36|              1692|         Valid|   SmartMeter|
|      1133|    M013|    W03|2026-10-14 09:00:00|    15.6|    

In [0]:
filtering_df = spark.table('water_meter_readings_source').filter(
    ((F.col('flow_lpm') >= 20) & (F.col('flow_lpm') <= 50)) &
    ((F.col('source_system') == 'Gateway') | (F.col('source_system') == 'SmartMeter'))
)

filtering_df.show()

+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+
|reading_id|meter_id|zone_id|       reading_time|flow_lpm|pressure_bar|chlorine_mg_l|consumption_liters|reading_status|source_system|
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+
|      1001|    M001|    W01|2026-10-01 07:00:00|    39.1|        3.56|         0.75|              2365|         Valid|      Gateway|
|      1014|    M014|    W04|2026-10-02 10:10:00|    22.4|        4.73|          0.5|              1108|         Valid|   SmartMeter|
|      1017|    M017|    W07|2026-10-02 13:10:00|    28.0|        3.77|         1.08|              1318|         Valid|   SmartMeter|
|      1019|    M019|    W09|2026-10-02 15:00:00|    23.0|        4.34|         0.28|              1091|         Valid|      Gateway|
|      1022|    M022|    W02|2026-10-03 08:00:00|    32.5|    

In [0]:
reading_status_df = spark.table('water_meter_readings_work').groupBy(
    'reading_status'
).agg(
    F.count('*').alias('count')
)

reading_status_df.show()

+--------------+-----+
|reading_status|count|
+--------------+-----+
|         Valid|  137|
|       Suspect|   11|
|      Rejected|    9|
+--------------+-----+



Section B | Joins, Aggregations and Window Functions

In [0]:
#6
readings_df = spark.table('water_meter_readings_work')
zones_df = spark.table('water_zones')
join_df = readings_df.join(
    zones_df,
    on='zone_id',
    how='inner'
).select(
    'zone_name',
    'district',
    'meter_id',
    'consumption_liters'
)

join_df.show()

+------------------+--------+--------+------------------+
|         zone_name|district|meter_id|consumption_liters|
+------------------+--------+--------+------------------+
|          Old City| Central|    M001|              2365|
|         Hill View|   North|    M002|              3545|
|   Market District| Central|    M003|               548|
|         Lake Side|    West|    M004|              2871|
|University Quarter|    East|    M005|              2656|
| Industrial Estate|   South|    M006|              2171|
|     Garden Colony|   North|    M007|              4167|
|       Transit Hub|    East|    M008|              3621|
|        Riverfront|    West|    M009|              1609|
|   South Extension|   South|    M010|              1829|
|          Old City| Central|    M011|               313|
|         Hill View|   North|    M012|              1303|
|   Market District| Central|    M013|              2071|
|         Lake Side|    West|    M014|              1108|
|University Qu

In [0]:
#7
unmatched_df = spark.table('water_meter_readings_work').join(
    spark.table('water_zones'),
    on='zone_id',
    how='left_anti'
)
unmatched_df.show()

+-------+----------+--------+-------------------+--------+------------+-------------+------------------+--------------+-------------+
|zone_id|reading_id|meter_id|       reading_time|flow_lpm|pressure_bar|chlorine_mg_l|consumption_liters|reading_status|source_system|
+-------+----------+--------+-------------------+--------+------------+-------------+------------------+--------------+-------------+
|    W99|      1154|    M999|2026-10-16 11:00:00|    24.5|         3.0|         0.45|              1150|         Valid|     FieldApp|
|   NULL|      1156|    M012|2026-10-16 12:30:00|    28.0|         3.1|         0.52|               870|         Valid|      Gateway|
+-------+----------+--------+-------------------+--------+------------+-------------+------------------+--------------+-------------+



In [0]:
#8
empty_zones_df = spark.table('water_zones').join(
    spark.table('water_meter_readings_work'),
    on='zone_id',
    how='left_anti'
)
empty_zones_df.show()

+-------+---------------+---------+--------+---------+---------------------+------+---------------+
|zone_id|      zone_name|     city|district|zone_type|daily_capacity_liters|active|commissioned_on|
+-------+---------------+---------+--------+---------+---------------------+------+---------------+
|    W11|New Development|Hyderabad|    West|    Mixed|               125000| false|     2021-03-15|
+-------+---------------+---------+--------+---------+---------------------+------+---------------+



In [0]:
#9
zone_summary_df = spark.table('water_meter_readings_work').filter(
    F.col('consumption_liters') > 0
).groupBy(
    'zone_id'
).agg(
    F.sum('consumption_liters').alias('total_consumption'),
    F.avg('pressure_bar').alias('avg_pressure'),
    F.count('*').alias('reading_count')
)

zone_summary_df.show()

+-------+-----------------+------------------+-------------+
|zone_id|total_consumption|      avg_pressure|reading_count|
+-------+-----------------+------------------+-------------+
|    W01|            27377| 3.437333333333333|           15|
|    W02|            27129|3.3573333333333335|           15|
|    W03|            39903|             3.178|           15|
|    W04|            25010|3.2829411764705885|           17|
|    W05|            31373|3.5933333333333333|           15|
|    W06|            25633|3.2250000000000005|           16|
|    W07|            29136|3.3706666666666667|           15|
|    W08|            37459|              2.92|           15|
|    W09|            26607|3.6040000000000005|           15|
|    W10|            31297|3.7079999999999997|           16|
|    W99|             1150|               3.0|            1|
|   NULL|              870|               3.1|            1|
+-------+-----------------+------------------+-------------+



In [0]:
#10
spark.table('water_meter_readings_work').createOrReplaceTempView('readings')
ranked_df = spark.sql("""
    select
        district,
        meter_id,
        total_consumption,
        rank() over (
            partition by district
            order by total_consumption desc
        ) as meter_rank
    from (
        select
            z.district,
            r.meter_id,
            sum(r.consumption_liters) as total_consumption
        from readings r
        join water_zones z
            on r.zone_id = z.zone_id
        where r.consumption_liters > 0
        group by z.district, r.meter_id
    )
""")

ranked_df.show()

+--------+--------+-----------------+----------+
|district|meter_id|total_consumption|meter_rank|
+--------+--------+-----------------+----------+
| Central|    M023|            17814|         1|
| Central|    M001|            11416|         2|
| Central|    M013|            11196|         3|
| Central|    M003|            10893|         4|
| Central|    M021|             8817|         5|
| Central|    M011|             7144|         6|
|    East|    M008|            14095|         1|
|    East|    M028|            12704|         2|
|    East|    M005|            11030|         3|
|    East|    M018|            10660|         4|
|    East|    M025|            10536|         5|
|    East|    M015|             9807|         6|
|   North|    M022|            12503|         1|
|   North|    M027|            10860|         2|
|   North|    M017|             9203|         3|
|   North|    M007|             9073|         4|
|   North|    M002|             8215|         5|
|   North|    M012| 

Section C | PySpark DataFrame Operations

In [0]:
#11
readings_df = spark.table('water_meter_readings_work')
zones_df = spark.table('water_zones')

readings_df.printSchema()

zones_df.printSchema()

root
 |-- reading_id: integer (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- reading_time: timestamp (nullable = true)
 |-- flow_lpm: double (nullable = true)
 |-- pressure_bar: double (nullable = true)
 |-- chlorine_mg_l: double (nullable = true)
 |-- consumption_liters: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- source_system: string (nullable = true)

root
 |-- zone_id: string (nullable = true)
 |-- zone_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- district: string (nullable = true)
 |-- zone_type: string (nullable = true)
 |-- daily_capacity_liters: long (nullable = true)
 |-- active: boolean (nullable = true)
 |-- commissioned_on: string (nullable = true)



In [0]:
#12
high_flow = readings_df.filter(
    (F.col('reading_status') == 'Valid') &
    (F.col('flow_lpm') > 50)
).select(
    'zone_id',
    'meter_id',
    'reading_time',
    'flow_lpm',
    'reading_status'
).orderBy(F.col('flow_lpm').desc())

high_flow.show()

+-------+--------+-------------------+--------+--------------+
|zone_id|meter_id|       reading_time|flow_lpm|reading_status|
+-------+--------+-------------------+--------+--------------+
|    W10|    M020|2026-10-11 16:10:00|    84.0|         Valid|
|    W03|    M023|2026-10-09 09:10:00|    83.9|         Valid|
|    W06|    M016|2026-10-11 12:00:00|    83.9|         Valid|
|    W10|    M010|2026-10-01 16:00:00|    81.3|         Valid|
|    W03|    M013|2026-10-02 09:00:00|    80.7|         Valid|
|    W08|    M018|2026-10-08 14:20:00|    80.7|         Valid|
|    W03|    M003|2026-10-10 09:20:00|    80.4|         Valid|
|    W01|    M001|2026-10-04 07:00:00|    80.1|         Valid|
|    W03|    M003|2026-10-13 09:20:00|    79.7|         Valid|
|    W04|    M014|2026-10-05 10:10:00|    79.3|         Valid|
|    W07|    M027|2026-10-12 13:20:00|    79.1|         Valid|
|    W06|    M026|2026-10-03 12:10:00|    79.1|         Valid|
|    W08|    M028|2026-10-09 14:00:00|    78.9|        

In [0]:
#13
flow_category_df = spark.table('water_meter_readings_work').withColumn(
    'flow_category',
    F.when(F.col('flow_lpm').isNull(), 'Unknown')
    .when(F.col('flow_lpm') >= 60, 'High')
    .when(F.col('flow_lpm') >= 30, 'Medium')
    .otherwise('Low')
)

flow_category_df.show()

+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+-------------+
|reading_id|meter_id|zone_id|       reading_time|flow_lpm|pressure_bar|chlorine_mg_l|consumption_liters|reading_status|source_system|flow_category|
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+-------------+
|      1001|    M001|    W01|2026-10-01 07:00:00|    39.1|        3.56|         0.75|              2365|         Valid|      Gateway|       Medium|
|      1002|    M002|    W02|2026-10-01 08:10:00|    59.9|         4.6|         0.39|              3545|         Valid|   SmartMeter|       Medium|
|      1003|    M003|    W03|2026-10-01 09:20:00|    25.7|        2.22|         1.15|               548|         Valid|     FieldApp|          Low|
|      1004|    M004|    W04|2026-10-01 10:00:00|    56.7|        2.84|          1.1|              2871|        

In [0]:
#14
readings_df = spark.table('water_meter_readings_work')
zones_df = spark.table('water_zones')

avg_flow_df = readings_df.join(
    zones_df,
    on='zone_id',
    how='inner'
).groupBy(
    'zone_type'
).agg(
    F.avg('flow_lpm').alias('avg_flow')
)

avg_flow_df.show()

+-------------+------------------+
|    zone_type|          avg_flow|
+-------------+------------------+
|   Commercial| 56.10000000000001|
|  Residential|45.155737704918025|
|        Mixed|         38.165625|
|   Industrial| 43.14374999999999|
|Institutional|          49.33125|
+-------------+------------------+



In [0]:
#15
top_zones_df = spark.table('water_meter_readings_work').filter(
    (F.col('reading_status') == 'Valid') &
    (F.col('consumption_liters') > 0)
).groupBy(
    'zone_id'
).agg(
    F.sum('consumption_liters').alias('total_consumption')
).orderBy(
    F.col('total_consumption').desc()
).limit(3)

top_zones_df.show()

+-------+-----------------+
|zone_id|total_consumption|
+-------+-----------------+
|    W03|            35665|
|    W08|            33689|
|    W05|            28717|
+-------+-----------------+



Section D | Data Quality and Cleaning

In [0]:
#16
duplicate_readings = readings_df.groupBy( 
    'reading_id'
).agg(
    F.count('*').alias('count')
).filter(
    F.col('count') > 1
)

duplicate_readings.show()

+----------+-----+
|reading_id|count|
+----------+-----+
|      1044|    2|
+----------+-----+



In [0]:
#17
missing_df = readings_df.filter(
    F.col('pressure_bar').isNull() |
    F.col('source_system').isNull()
)

missing_df.show()

print('missing records:', missing_df.count())

+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+
|reading_id|meter_id|zone_id|       reading_time|flow_lpm|pressure_bar|chlorine_mg_l|consumption_liters|reading_status|source_system|
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+
|      1153|    M020|    W10|2026-10-16 10:00:00|    31.4|        NULL|         0.71|              1020|         Valid|      Gateway|
|      1155|    M006|    W06|2026-10-16 12:00:00|    44.0|         2.9|         2.95|              1560|       Suspect|         NULL|
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+

missing records: 2


In [0]:
#18
invalid_df = readings_df.filter(
    (F.col('flow_lpm') < 0) |
    (F.col('consumption_liters') <= 0) |
    (F.col('zone_id').isNull())
)

invalid_df.show()

+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+
|reading_id|meter_id|zone_id|       reading_time|flow_lpm|pressure_bar|chlorine_mg_l|consumption_liters|reading_status|source_system|
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+
|      1151|    M004|    W04|2026-10-16 08:00:00|   -12.5|         2.8|         0.62|               500|       Suspect|      Gateway|
|      1152|    M015|    W05|2026-10-16 09:00:00|    22.0|         2.1|          0.5|                 0|         Valid|   SmartMeter|
|      1156|    M012|   NULL|2026-10-16 12:30:00|    28.0|         3.1|         0.52|               870|         Valid|      Gateway|
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+



In [0]:
#19
from pyspark.sql.window import Window
window_spec = Window.partitionBy('reading_id').orderBy(F.col('reading_time').desc())

clean_df = readings_df.filter(
    (F.col('reading_status') == 'Valid') &
    (F.col('flow_lpm') > 0) &
    (F.col('consumption_liters') > 0) &
    (F.col('zone_id').isNotNull())
).withColumn(
    'rn',
    F.row_number().over(window_spec)
).filter(
    F.col('rn') == 1
).drop('rn')

clean_df.write.format('delta').mode('overwrite').saveAsTable(
    'water_readings_clean'
)

display(spark.table('water_readings_clean'))

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1001,M001,W01,2026-10-01T07:00:00.000Z,39.1,3.56,0.75,2365,Valid,Gateway
1002,M002,W02,2026-10-01T08:10:00.000Z,59.9,4.6,0.39,3545,Valid,SmartMeter
1003,M003,W03,2026-10-01T09:20:00.000Z,25.7,2.22,1.15,548,Valid,FieldApp
1004,M004,W04,2026-10-01T10:00:00.000Z,56.7,2.84,1.1,2871,Valid,Gateway
1006,M006,W06,2026-10-01T12:20:00.000Z,35.8,2.52,1.2,2171,Valid,FieldApp
1007,M007,W07,2026-10-01T13:00:00.000Z,65.0,3.19,0.67,4167,Valid,Gateway
1008,M008,W08,2026-10-01T14:10:00.000Z,70.3,3.39,0.54,3621,Valid,SmartMeter
1009,M009,W09,2026-10-01T15:20:00.000Z,30.8,4.57,1.05,1609,Valid,FieldApp
1010,M010,W10,2026-10-01T16:00:00.000Z,81.3,2.38,0.33,1829,Valid,Gateway
1011,M011,W01,2026-10-02T07:10:00.000Z,13.0,4.39,0.66,313,Valid,SmartMeter


In [0]:
#20
requests_df = spark.table('water_service_requests')

high_priority_df = requests_df.filter(
    F.col('priority') == 'High'
).select(
    'zone_id'
).distinct()

pressure_df = clean_df.join(
    high_priority_df,
    on='zone_id',
    how='inner'
).groupBy(
    'zone_id'
).agg(
    F.avg('pressure_bar').alias('avg_pressure')
)

pressure_df.show()

+-------+------------------+
|zone_id|      avg_pressure|
+-------+------------------+
|    W01| 3.423076923076923|
|    W02|3.3061538461538458|
|    W03| 3.019999999999999|
|    W04| 3.336153846153846|
|    W06| 3.160769230769231|
|    W07|3.5700000000000003|
|    W08|2.8657142857142857|
|    W09|3.5469230769230773|
|    W10| 3.717692307692307|
|    W05| 3.632857142857143|
+-------+------------------+



Section E | Delta Lake Tables and CRUD

In [0]:
%sql
--21
create or replace table water_readings_work_copy
using delta
as 
select *
from water_meter_readings_work;

num_affected_rows,num_inserted_rows


In [0]:
%sql
--22
insert into water_readings_work_copy
values
(2001, 101, 1, current_timestamp(), 35.5, 4.2, 0.5, 250, 'Valid', 'Gateway'),
(2002, 102, 2, current_timestamp(), 42.0, 3.8, 0.6, 300, 'Valid', 'SmartMeter');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
--23
select *
from water_readings_work_copy
where reading_status = 'Suspect'
limit 5;
update water_readings_work_copy
set reading_status = 'Rejected'
where reading_id = 1005
and reading_status = 'Suspect';

num_affected_rows
1


In [0]:
%sql
--24
select count(*) as before_count
from water_readings_work_copy;
delete from water_readings_work_copy
where reading_status = 'Rejected';
select count(*) as after_count
from water_readings_work_copy;

after_count
149


In [0]:
#25
zone_summary_df = spark.table('water_readings_clean').groupBy(
    'zone_id'
).agg(
    F.sum('consumption_liters').alias('total_consumption'),
    F.avg('pressure_bar').alias('avg_pressure'),
    F.count('*').alias('reading_count')
)
zone_summary_df.write.format('delta').mode('overwrite').saveAsTable(
    'zone_consumption_summary'
)

display(spark.table('zone_consumption_summary'))

zone_id,total_consumption,avg_pressure,reading_count
W06,20212,3.160769230769231,13
W10,27808,3.717692307692307,14
W07,25104,3.5700000000000003,13
W08,33689,2.8657142857142857,14
W02,24120,3.3061538461538458,13
W03,35665,3.019999999999999,13
W99,1150,3.0,1
W04,18120,3.336153846153846,13
W01,23107,3.423076923076923,13
W09,20894,3.5469230769230773,13


Section F | MERGE, UPSERT and SCD

In [0]:
%sql
--26
--matching
select c.*
from water_reading_changes c
inner join water_meter_readings_work w
on c.reading_id = w.reading_id;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1005,M005,W05,2026-10-01T11:10:00.000Z,64.3,3.24,0.43,2776,Valid,FieldApp
1027,M027,W07,2026-10-03T13:20:00.000Z,54.6,4.99,0.95,2678,Valid,FieldApp
1062,M002,W02,2026-10-07T08:10:00.000Z,30.1,2.43,0.3,1057,Valid,FieldApp
1088,M028,W08,2026-10-09T14:00:00.000Z,81.4,2.17,1.06,4931,Valid,FieldApp
1119,M029,W09,2026-10-12T15:10:00.000Z,22.1,3.63,1.13,1339,Valid,FieldApp
1145,M025,W05,2026-10-15T11:00:00.000Z,77.5,3.96,1.29,4908,Valid,FieldApp


In [0]:
%sql
--new
select c.*
from water_reading_changes c
left anti join water_meter_readings_work w
on c.reading_id = w.reading_id;

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1201,M002,W02,2026-10-18T08:00:00.000Z,24.0,2.6,0.45,1100,Valid,FieldApp
1202,M005,W05,2026-10-18T09:00:00.000Z,28.7,2.7,0.5,1250,Valid,FieldApp
1203,M008,W08,2026-10-18T10:00:00.000Z,33.4,2.8,0.55,1400,Valid,FieldApp
1204,M011,W01,2026-10-18T11:00:00.000Z,38.1,2.9,0.6,1550,Valid,FieldApp
1205,M014,W04,2026-10-18T12:00:00.000Z,42.8,3.0,0.65,1700,Valid,FieldApp
1206,M017,W07,2026-10-18T13:00:00.000Z,47.5,3.1,0.7,1850,Valid,FieldApp


In [0]:
%sql
--27
merge into water_meter_readings_work as w
using water_reading_changes as c
on w.reading_id = c.reading_id

when matched then
    update set *

when not matched then
    insert *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
12,6,0,6


In [0]:
%sql
--28 
--update checking
select *
from water_meter_readings_work
where reading_id in (
    select reading_id
    from water_reading_changes
);

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1005,M005,W05,2026-10-01T11:10:00.000Z,64.3,3.24,0.43,2776,Valid,FieldApp
1027,M027,W07,2026-10-03T13:20:00.000Z,54.6,4.99,0.95,2678,Valid,FieldApp
1062,M002,W02,2026-10-07T08:10:00.000Z,30.1,2.43,0.3,1057,Valid,FieldApp
1088,M028,W08,2026-10-09T14:00:00.000Z,81.4,2.17,1.06,4931,Valid,FieldApp
1119,M029,W09,2026-10-12T15:10:00.000Z,22.1,3.63,1.13,1339,Valid,FieldApp
1145,M025,W05,2026-10-15T11:00:00.000Z,77.5,3.96,1.29,4908,Valid,FieldApp
1201,M002,W02,2026-10-18T08:00:00.000Z,24.0,2.6,0.45,1100,Valid,FieldApp
1202,M005,W05,2026-10-18T09:00:00.000Z,28.7,2.7,0.5,1250,Valid,FieldApp
1203,M008,W08,2026-10-18T10:00:00.000Z,33.4,2.8,0.55,1400,Valid,FieldApp
1204,M011,W01,2026-10-18T11:00:00.000Z,38.1,2.9,0.6,1550,Valid,FieldApp


In [0]:
%sql
create or replace table water_meter_readings_work_before
as
select * from water_meter_readings_work;
create or replace table water_meter_readings_work_pyspark
as
select * from water_meter_readings_work_before;

num_affected_rows,num_inserted_rows


In [0]:
#29
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark, 'water_meter_readings_work_pyspark'
)
source = spark.table('water_reading_changes')

target.alias('t').merge(
    source.alias('s'),
    't.reading_id = s.reading_id'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

display(spark.table('water_meter_readings_work_pyspark'))

reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1001,M001,W01,2026-10-01T07:00:00.000Z,39.1,3.56,0.75,2365,Valid,Gateway
1002,M002,W02,2026-10-01T08:10:00.000Z,59.9,4.6,0.39,3545,Valid,SmartMeter
1003,M003,W03,2026-10-01T09:20:00.000Z,25.7,2.22,1.15,548,Valid,FieldApp
1004,M004,W04,2026-10-01T10:00:00.000Z,56.7,2.84,1.1,2871,Valid,Gateway
1006,M006,W06,2026-10-01T12:20:00.000Z,35.8,2.52,1.2,2171,Valid,FieldApp
1007,M007,W07,2026-10-01T13:00:00.000Z,65.0,3.19,0.67,4167,Valid,Gateway
1008,M008,W08,2026-10-01T14:10:00.000Z,70.3,3.39,0.54,3621,Valid,SmartMeter
1009,M009,W09,2026-10-01T15:20:00.000Z,30.8,4.57,1.05,1609,Valid,FieldApp
1010,M010,W10,2026-10-01T16:00:00.000Z,81.3,2.38,0.33,1829,Valid,Gateway
1011,M011,W01,2026-10-02T07:10:00.000Z,13.0,4.39,0.66,313,Valid,SmartMeter


In [0]:
%sql
--30
--create
create or replace table zone_capacity_history
using delta
as
select
    zone_id,
    zone_name,
    district,
    daily_capacity_liters,
    zone_type,
    date'1900-01-01' as valid_from,
    cast(null as date) as valid_to,
    true as is_current
from water_zones;

num_affected_rows,num_inserted_rows


In [0]:
%sql
--update
update zone_capacity_history
set
    valid_to = date'2026-10-19',
    is_current = false
where zone_id in ('W02', 'W06')
and is_current = true;

num_affected_rows
2


In [0]:
%sql
insert into zone_capacity_history (
    zone_id,
    zone_name,
    district,
    zone_type,
    daily_capacity_liters,
    valid_from,
    valid_to,
    is_current
)
select
    zone_id,
    zone_name,
    district,
    zone_type,
    case
        when zone_id = 'W02' then 195000
        when zone_id = 'W06' then 305000
    end,
    date'2026-10-20',
    cast(null as date),
    true
from water_zones
where zone_id in ('W02', 'W06');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
--verify
select *
from zone_capacity_history
where zone_id in ('W02', 'W06')
order by zone_id, valid_from;

zone_id,zone_name,district,daily_capacity_liters,zone_type,valid_from,valid_to,is_current
W02,Hill View,North,180000,Residential,1900-01-01,2026-10-19,false
W02,Hill View,North,195000,Residential,2026-10-20,null,true
W06,Industrial Estate,South,320000,Industrial,1900-01-01,2026-10-19,false
W06,Industrial Estate,South,305000,Industrial,2026-10-20,null,true


Section G | Delta History, Time Travel and Maintenance

In [0]:
%sql
--31
describe detail water_meter_readings_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,92dec5f3-0618-4349-beb7-8f0ee3f395a4,main.default.water_meter_readings_work,null,,2026-11-11T16:36:37.852Z,2026-11-11T22:12:22.000Z,List(),List(),2,8837,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 6, numDeletionVectors -> 1)",false


In [0]:
%sql
--32
describe history water_meter_readings_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-11-11T22:12:22.000Z,1316548390739548,dharshinis369@gmail.com,MERGE,"Map(predicate -> [""(reading_id#17593 = reading_id#17613)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3413288353911961),49fd3996-1383-44bb-879f-85f526a5baf9,1111-212517-tsi0y20-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3413, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 6, executionTimeMs -> 2946, materializeSourceTimeMs -> 6, numTargetRowsInserted -> 6, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1011, numTargetRowsUpdated -> 6, numOutputRows -> 12, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 12, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1863)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-11T16:36:40.000Z,1316548390739548,dharshinis369@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3413288353911961),2d6a2f71-e390-424e-8f8c-4d364d7d1860,1111-150832-wql6kysm-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 157, numOutputBytes -> 5424)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13


In [0]:
%sql
--33
select 'before' as operation,*
from water_meter_readings_work version as of 0
where reading_id = 1005
union all 
select 'after' as operation,*
from water_meter_readings_work
where reading_id = 1005;

operation,reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
before,1005,M005,W05,2026-10-01T11:10:00.000Z,61.8,3.04,0.43,2656,Suspect,SmartMeter
after,1005,M005,W05,2026-10-01T11:10:00.000Z,64.3,3.24,0.43,2776,Valid,FieldApp


In [0]:
%sql
--34
optimize water_meter_readings_work;

    

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1794436116074, 1794436116587, 8, 0, null, List(0, 0), null, 10, 10, 0, 0, null, null, 0)"


In [0]:
%sql
--35
vacuum water_meter_readings_work dry run;
--Vacuum will remove unused files. The default retention period is 7 days. The histrical files will be affected.

path


Section H | Unity Catalog, Views and Parquet

In [0]:
%sql
--36
select current_catalog(), current_schema();

current_catalog(),current_schema()
main,default


In [0]:
%sql
show tables;

database,tableName,isTemporary
default,attraction_capacity_history,false
default,bronze_attractions,false
default,bronze_entries,false
default,bronze_layer,false
default,bronze_maintenance,false
default,bronze_stream,false
default,glod_layer,false
default,gold,false
default,gold_daily_summary,false
default,permission_test,false


In [0]:
%sql
describe detail water_meter_readings_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,92dec5f3-0618-4349-beb7-8f0ee3f395a4,main.default.water_meter_readings_work,null,,2026-11-11T16:36:37.852Z,2026-11-11T22:28:24.000Z,List(),List(),1,5838,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
--37
create or replace view valid_readings 
as
select r.*,z.zone_name,z.district  
from water_meter_readings_work r
inner join water_zones z
on r.zone_id = z.zone_id
where reading_status = 'Valid'


In [0]:
%sql
--38
create or replace temp view low_pressure 
as
select *
from water_meter_readings_work 
where pressure_bar < 3;

In [0]:
%sql
--39
create or replace global temporary view high_priority_requests_view 
as
select *
from water_service_requests
where priority = 'High'

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8949792006633413>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "--39\ncreate or replace global temporary view high_priority_requests_view \nas\nselect *\nfrom water_service_requests\nwhere priority = 'High'\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:217, in S

In [0]:
#40
readings_df = spark.table('water_readings_clean')
path = '/Volumes/main/default/assessment_files/water_readings_parquet'

readings_df.write.mode('overwrite').parquet(path)
parquet_df = spark.read.parquet(path)

parquet_df.show()
parquet_df.write.format('delta').mode('overwrite').saveAsTable(
    'water_readings_from_parquet'
)


+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+
|reading_id|meter_id|zone_id|       reading_time|flow_lpm|pressure_bar|chlorine_mg_l|consumption_liters|reading_status|source_system|
+----------+--------+-------+-------------------+--------+------------+-------------+------------------+--------------+-------------+
|      1001|    M001|    W01|2026-10-01 07:00:00|    39.1|        3.56|         0.75|              2365|         Valid|      Gateway|
|      1002|    M002|    W02|2026-10-01 08:10:00|    59.9|         4.6|         0.39|              3545|         Valid|   SmartMeter|
|      1003|    M003|    W03|2026-10-01 09:20:00|    25.7|        2.22|         1.15|               548|         Valid|     FieldApp|
|      1004|    M004|    W04|2026-10-01 10:00:00|    56.7|        2.84|          1.1|              2871|         Valid|      Gateway|
|      1006|    M006|    W06|2026-10-01 12:20:00|    35.8|    

Section I | Structured Streaming

In [0]:
#41
streaming_df = spark.readStream.format('rate').option('rowsPerSecond', 5).load()

streaming_df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)



In [0]:
#42
from pyspark.sql import functions as F
simulated_meter = streaming_df.withColumn(
    'meter_id',
    (F.col('value') % 100 )+1
).withColumn(
    'flow_lpm',
    (F.col('value')%80)+10
)


In [0]:
# 43
filtered_df = simulated_meter.filter(
    F.col('flow_lpm') > 50
)

filter_query = filtered_df.writeStream.format('console').outputMode(
    'append'
).option('truncate', False).option(
    'checkpointLocation',
    '/Volumes/main/default/assessment_files/checkpoints/high_flow_console'
).trigger(
    availableNow=True
).start()

In [0]:
#44
from pyspark.sql.functions import when, count

category_df = simulated_meter.withColumn(
    'flow_category',
    when(F.col('flow_lpm') >= 60, 'High')
    .when(F.col('flow_lpm') >= 30, 'Medium')
    .otherwise('Low')
)
count_df = category_df.groupBy('flow_category').agg(
    count('*').alias('total_readings')
)

count_query = count_df.writeStream.format('console').outputMode(
    'complete'
).option('truncate', False).option(
    'checkpointLocation',
    '/Volumes/main/default/assessment_files/checkpoints/flow_category_counts'
).trigger(
    availableNow=True
).start()

In [0]:
#45
count_query.stop()
'''Checkpoint store the streaming query progrss and state information. Track processing of data and avoid reprocessing of data that has already been processed'''

Section K | Incremental Loads, Analytics and Visualizations

In [0]:
%sql
--51
describe water_readings_incremental_staging;
describe water_meter_readings_source;

col_name,data_type,comment
reading_id,int,null
meter_id,string,null
zone_id,string,null
reading_time,timestamp,null
flow_lpm,double,null
pressure_bar,double,null
chlorine_mg_l,double,null
consumption_liters,int,null
reading_status,string,null
source_system,string,null


In [0]:
%sql
select count(*) as staging_count
from water_readings_incremental_staging;

staging_count
18


In [0]:
%sql
insert into water_meter_readings_source
select s.*
from water_readings_incremental_staging s
left anti join water_meter_readings_source t
on s.reading_id = t.reading_id;

num_affected_rows,num_inserted_rows
18,18


In [0]:
%sql
--52
select *
from gold_assessmanet2
order by zone_id;

zone_id,total_consumption,average_pressure,total_readings
null,870,3.1,1
W01,23107,3.423076923076923,13
W02,24120,3.3061538461538458,13
W03,35665,3.019999999999999,13
W04,20781,3.2371428571428575,14
W05,28717,3.632857142857143,14
W06,20212,3.160769230769231,13
W07,25104,3.5700000000000003,13
W08,33689,2.8657142857142857,14
W09,20894,3.5469230769230773,13


In [0]:
%sql
--53
select
    zone_id,
    sum(consumption_liters) as total_consumption
from silver_stream_table
group by zone_id
order by total_consumption desc;

zone_id,total_consumption
W03,35665
W08,33689
W05,28717
W10,27808
W07,25104
W02,24120
W01,23107
W09,20894
W04,20781
W06,20212


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
select
    z.district,
    sum(r.consumption_liters) as total_consumption
from silver_stream_table r
join water_zones z
on r.zone_id = z.zone_id
group by z.district
order by total_consumption desc;

district,total_consumption
East,62406
Central,58772
North,49224
South,48020
West,41675


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
--54
select
    to_date(reading_time) as reading_date,
    sum(consumption_liters) as daily_consumption
from silver_stream_table
group by to_date(reading_time)
order by reading_date;

reading_date,daily_consumption
2026-10-01,22726
2026-10-02,13169
2026-10-03,14187
2026-10-04,19704
2026-10-05,16271
2026-10-06,19793
2026-10-07,11012
2026-10-08,14937
2026-10-09,19235
2026-10-10,16787


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
select
    pressure_bar,
    flow_lpm
from silver_stream_table
where pressure_bar is not null
and flow_lpm is not null;

pressure_bar,flow_lpm
3.56,39.1
4.6,59.9
2.22,25.7
2.84,56.7
2.52,35.8
3.19,65.0
3.39,70.3
4.57,30.8
2.38,81.3
4.39,13.0


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
--55
select
    zone_id,
    sum(consumption_liters) as total_consumption
from silver_stream_table
group by zone_id
order by total_consumption desc;

zone_id,total_consumption
W03,35665
W08,33689
W05,28717
W10,27808
W07,25104
W02,24120
W01,23107
W09,20894
W04,20781
W06,20212


Final capstone

In [0]:
%sql
-- 1. Zone-wise summary
select
    zone_id,
    sum(consumption_liters) as total_consumption,
    avg(pressure_bar) as average_pressure,
    avg(chlorine_mg_l) as average_chlorine
from silver_stream_table
group by zone_id;

zone_id,total_consumption,average_pressure,average_chlorine
W01,23107,3.423076923076923,0.8200000000000002
W02,24120,3.3061538461538458,0.6192307692307693
W03,35665,3.019999999999999,0.936153846153846
W04,20781,3.2371428571428575,0.817857142857143
W06,20212,3.160769230769231,0.6776923076923076
W07,25104,3.5700000000000003,0.773076923076923
W08,33689,2.8657142857142857,0.6185714285714284
W09,20894,3.5469230769230773,0.6815384615384615
W10,27808,3.717692307692307,0.7928571428571427
W05,28717,3.632857142857143,0.8735714285714286


In [0]:
%sql
--Historical Delta query
select *
from water_meter_readings_work
version as of 0;


reading_id,meter_id,zone_id,reading_time,flow_lpm,pressure_bar,chlorine_mg_l,consumption_liters,reading_status,source_system
1001,M001,W01,2026-10-01T07:00:00.000Z,39.1,3.56,0.75,2365,Valid,Gateway
1002,M002,W02,2026-10-01T08:10:00.000Z,59.9,4.6,0.39,3545,Valid,SmartMeter
1003,M003,W03,2026-10-01T09:20:00.000Z,25.7,2.22,1.15,548,Valid,FieldApp
1004,M004,W04,2026-10-01T10:00:00.000Z,56.7,2.84,1.1,2871,Valid,Gateway
1005,M005,W05,2026-10-01T11:10:00.000Z,61.8,3.04,0.43,2656,Suspect,SmartMeter
1006,M006,W06,2026-10-01T12:20:00.000Z,35.8,2.52,1.2,2171,Valid,FieldApp
1007,M007,W07,2026-10-01T13:00:00.000Z,65.0,3.19,0.67,4167,Valid,Gateway
1008,M008,W08,2026-10-01T14:10:00.000Z,70.3,3.39,0.54,3621,Valid,SmartMeter
1009,M009,W09,2026-10-01T15:20:00.000Z,30.8,4.57,1.05,1609,Valid,FieldApp
1010,M010,W10,2026-10-01T16:00:00.000Z,81.3,2.38,0.33,1829,Valid,Gateway


In [0]:
%sql
-- bar chart - zone consumption
select
    zone_id,
    sum(consumption_liters) as total_consumption
from silver_stream_table
group by zone_id;

zone_id,total_consumption
W01,23107
W02,24120
W03,35665
W04,20781
W06,20212
W07,25104
W08,33689
W09,20894
W10,27808
W05,28717
